# Problem 2 - Wine Quality Prediction on AWS SageMaker

In [1]:
import pandas as pd
import numpy as np
import boto3
import sagemaker
from sagemaker import get_execution_role

# Initialize SageMaker

session = sagemaker.Session()
role = get_execution_role()
bucket = session.default_bucket()
prefix = 'sagemaker/wine-quality'

# Read Datasets

red_df = pd.read_csv('shared/winequality-red.csv', sep=';')
white_df = pd.read_csv('shared/winequality-white.csv', sep=';')

# Combine Datasets
df = pd.concat([red_df, white_df], axis=0, ignore_index=True)

# Reorder columns

df_sm = pd.concat([df['quality'], df.drop(columns=['quality'])], axis=1)

# Split data 80/20

train_data, val_data = np.split(df_sm.sample(frac=1, random_state=42), [int(0.8 * len(df_sm))])

# Export
train_data.to_csv('train.csv', header=False, index=False)
val_data.to_csv('validation.csv', header=False, index=False)

# Upload Training and validation to S3
train_s3 = session.upload_data('train.csv', bucket=bucket, key_prefix=f'{prefix}/train')
val_s3 = session.upload_data('validation.csv', bucket=bucket, key_prefix=f'{prefix}/val')

print(f"Data Uploaded to S3 Bucket: {bucket}")

sagemaker.config INFO - Fetched defaults config from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
Data Uploaded to S3 Bucket: amazon-sagemaker-365698175161-us-east-2-cu2ccb8aeza2sp


## Part (b) - Linear Regression WITH Conatiner Technology in SageMaker

In [7]:
from sagemaker import image_uris
from sagemaker.inputs import TrainingInput
from sagemaker.estimator import Estimator


#Retrieve container
container = image_uris.retrieve('linear-learner', session.boto_region_name)

#Define Estimator
linear_estimator = Estimator(
    image_uri=container,
    role=role,
    instance_count=1,
    instance_type='ml.m5.large',
    output_path=f's3://{bucket}/{prefix}/output',
    sagemaker_session=session
)

# Set Hyperparameters

linear_estimator.set_hyperparameters(
    feature_dim=11,
    predictor_type='regressor',
    mini_batch_size=32
)

# Define S3 channels

s3_train = TrainingInput(s3_data=train_s3, content_type='text/csv')
s3_val = TrainingInput(s3_data=val_s3, content_type='text/csv')

# Train Model

linear_estimator.fit({'train': s3_train, 'validation': s3_val})

# Deploy
predictor_a = linear_estimator.deploy(
    initial_instance_count=1,
    instance_type='ml.m5.large'
)


print("Deployed Successfully")


sagemaker.config INFO - Applied value from config key = SageMaker.TrainingJob.Environment


╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:34                                                                                   │
│                                                                                                  │
│   31                                                                                             │
│   32 # Train Model                                                                               │
│   33                                                                                             │
│ ❱ 34 linear_estimator.fit({'train': s3_train, 'validation': s3_val})                             │
│   35                                                                                             │
│   36 # Deploy                                                                                    │
│   37 predictor_a = linear_estimator.deploy(                                                      │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/telemetry/telemetry_logging.py:171 in wrapper  │
│                                                                                                  │
│   168 │   │   │   │   │   caught_ex = e                                                          │
│   169 │   │   │   │   finally:                                                                   │
│   170 │   │   │   │   │   if caught_ex:                                                          │
│ ❱ 171 │   │   │   │   │   │   raise caught_ex                                                    │
│   172 │   │   │   │   │   return response  # pylint: disable=W0150                               │
│   173 │   │   │   else:                                                                          │
│   174 │   │   │   │   logger.debug(                                                              │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/telemetry/telemetry_logging.py:142 in wrapper  │
│                                                                                                  │
│   139 │   │   │   │   start_timer = perf_counter()                                               │
│   140 │   │   │   │   try:                                                                       │
│   141 │   │   │   │   │   # Call the original function                                           │
│ ❱ 142 │   │   │   │   │   response = func(*args, **kwargs)                                       │
│   143 │   │   │   │   │   stop_timer = perf_counter()                                            │
│   144 │   │   │   │   │   elapsed = stop_timer - start_timer                                     │
│   145 │   │   │   │   │   extra += f"&x-latency={round(elapsed, 2)}"                             │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/workflow/pipeline_context.py:346 in wrapper    │
│                                                                                                  │
│   343 │   │   │                                                                                  │
│   344 │   │   │   return _StepArguments(retrieve_caller_name(self_instance), run_func, *args,    │
│   345 │   │                                                                                      │
│ ❱ 346 │   │   return run_func(*args, **kwargs)                                                   │
│   347 │                                                                                          │
│   348 │   return wrapper                                                                         │
│   349                                                      

## Part (a) - Linear Regression WITHOUT cContainer Technology

In [5]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Train and val

X_train = train_data.iloc[:, 1:]
y_train = train_data.iloc[:, 0]
X_val = val_data.iloc[:, 1:]
y_val = val_data.iloc[:,0]

lr = LinearRegression()
lr.fit(X_train, y_train)
preds = lr.predict(X_val)

print("R Squared:", r2_score(y_val, preds))
print("RMSE:", mean_squared_error(y_val, preds) ** 0.5)

R Squared: 0.2837322038979109
RMSE: 0.7379489530808739
